# SafeRest360 — Demo Interactiva

Demo de detección de personas + zonas + reglas.
Sube una imagen y observa el pipeline completo.

In [ ]:
!pip install -q ultralytics supervision opencv-python-headless

import sys
sys.path.insert(0, 'SafeRest360/vision-engine')

In [ ]:
from src.inference.detector import Detector
from src.inference.tracker import PersonTracker
from src.rules.engine import RulesEngine, RuleConfig
from src.rules.zone import ZonePolygon
from src.utils.visualization import draw_detections, draw_zones
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Usar modelo pretrained (ya sabe detectar personas desde COCO)
detector = Detector(model_path='yolov8n.pt')
tracker = PersonTracker()
print("Modelo cargado.")

In [ ]:
# Configurar zonas de ejemplo
restricted_zone = ZonePolygon(
    zone_id='z1', name='Cocina', zone_type='restricted',
    polygon_points=[
        {'x': 0.0, 'y': 0.0}, {'x': 0.3, 'y': 0.0},
        {'x': 0.3, 'y': 1.0}, {'x': 0.0, 'y': 1.0}
    ]
)

monitored_zone = ZonePolygon(
    zone_id='z2', name='Caja', zone_type='monitored',
    polygon_points=[
        {'x': 0.7, 'y': 0.0}, {'x': 1.0, 'y': 0.0},
        {'x': 1.0, 'y': 1.0}, {'x': 0.7, 'y': 1.0}
    ]
)

# Motor de reglas
engine = RulesEngine()
engine.add_rule(RuleConfig(
    rule_id='r1', rule_type='no_entry', zone=restricted_zone,
    camera_id='cam1', parameters={}, severity='high', cooldown_seconds=0
))
engine.add_rule(RuleConfig(
    rule_id='r2', rule_type='zone_abandoned', zone=monitored_zone,
    camera_id='cam1', parameters={'timeout_seconds': 30},
    severity='medium', cooldown_seconds=0
))
print("Zonas y reglas configuradas.")

In [ ]:
# Subir imagen o usar una de prueba
from google.colab import files
uploaded = files.upload()

for filename in uploaded:
    img = cv2.imread(filename)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Detectar personas
    raw_dets = detector.detect_raw(img_rgb, confidence_threshold=0.4)
    tracked = tracker.update(raw_dets)
    
    # Evaluar reglas
    det_dicts = [d.to_dict() for d in tracked]
    h, w = img.shape[:2]
    events = engine.evaluate(det_dicts, frame_width=w, frame_height=h)
    
    # Visualizar
    vis = img_rgb.copy()
    zones = [restricted_zone, monitored_zone]
    vis = draw_zones(vis, [{'polygon_points': z.polygon_points, 'name': z.name,
                            'color': '#ef4444' if z.zone_type == 'restricted' else '#f59e0b'}
                           for z in zones])
    vis = draw_detections(vis, tracked)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(vis)
    plt.axis('off')
    plt.title(f'Personas: {len(tracked)} | Eventos: {len(events)}')
    plt.show()
    
    if events:
        print("\n--- ALERTAS ---")
        for e in events:
            print(f"  [{e.severity.upper()}] {e.event_type}: {e.description}")
    else:
        print("\nSin alertas.")